# Loss landscapes around truth (HK multi-wavelength closure)

Pairs of fit parameters against each other, as in the LUCiD calibration slides: each axis is a
parameter's scale relative to truth, the colour is the local Δloss (log scale), and the white
streamlines run down the gradient towards the panel's minimum. The loss is the closure fit's own:
the cross χ² (Σ r_A·r_B over two independent forward draws, so no Monte-Carlo variance term),
gains profiled as the fit does, every other parameter at truth. Truth is the star, the fit the
cross, its path the thin line.

Settings (pairs, spans, grid, rays) are constants at the top of `hk_landscape_job.py`; it reads the
closure run's results file, so point `--results` at the run to map. This notebook only loads and plots.

In [ ]:
import sys

!cd .. && {sys.executable} hk_landscape_job.py --gpu 5 --results results/hk_new.npz --out results/hk_landscape.npz

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

L = np.load('../results/hk_landscape.npz')
names, truth = list(L['names']), L['truth']
idx = {n: i for i, n in enumerate(names)}
fit = L['fit'] if 'fit' in L.files else None
hist = L['history'] if 'history' in L.files else None
pairs = [tuple(p) for p in L['pairs']]
label = lambda n: n.replace('_', ' ')

deltas = [g - g.min() for g in L['loss']]
positive = np.concatenate([d[d > 0] for d in deltas])
norm = LogNorm(vmin=np.percentile(positive, 1), vmax=positive.max())

ncol = 3
nrow = int(np.ceil(len(pairs) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.6 * ncol, 4.8 * nrow), squeeze=False,
                         layout='constrained')
print(f'{"pair":44s}{"minimum at (x, y) scale":>26s}{"loss(truth) - min":>20s}')
for ax, (a, b), sa, sb, grid, d in zip(axes.flat, pairs, L['scale_x'], L['scale_y'], L['loss'], deltas):
    mesh = ax.pcolormesh(sa, sb, np.clip(d, norm.vmin, None), norm=norm, cmap='viridis',
                         shading='gouraud')
    gy, gx = np.gradient(grid, sb, sa)
    ax.streamplot(sa, sb, -gx, -gy, color='white', density=1.1, linewidth=0.7, arrowsize=0.8)
    if hist is not None:
        ax.plot(hist[:, idx[a]] / truth[idx[a]], hist[:, idx[b]] / truth[idx[b]],
                color='cyan', lw=0.8, alpha=0.7)
    ax.plot(1, 1, marker='*', ms=15, color='deeppink', mec='white', ls='', label='truth')
    if fit is not None:
        ax.plot(fit[idx[a]] / truth[idx[a]], fit[idx[b]] / truth[idx[b]], marker='X', ms=11,
                color='cyan', mec='k', ls='', label='fit')
    iy, ix = np.unravel_index(np.argmin(grid), grid.shape)
    ax.plot(sa[ix], sb[iy], marker='o', ms=5, color='white', ls='', label='panel minimum')
    ax.set_xlim(sa[0], sa[-1]); ax.set_ylim(sb[0], sb[-1])
    ax.set_xlabel(f'{label(a)} scale'); ax.set_ylabel(f'{label(b)} scale')
    ax.set_title(f'{label(a)} vs {label(b)}')
    i1 = np.argmin(np.abs(sa - 1)); j1 = np.argmin(np.abs(sb - 1))
    print(f'{a + " x " + b:44s}{f"({sa[ix]:.3f}, {sb[iy]:.3f})":>26s}{grid[j1, i1] - grid.min():20.1f}')
for ax in axes.flat[len(pairs):]:
    ax.set_visible(False)
axes.flat[0].legend(loc='upper left', fontsize=8, framealpha=0.7)
fig.colorbar(mesh, ax=axes, label='local Δloss (cross χ²)', shrink=0.8)
fig.suptitle(f'loss landscape around truth ({L["results"]}, {int(L["n_rays"]):,} rays)', fontweight='bold')
plt.show()